In [ ]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime
import logging
import sys

import os


# Set up project path for Jupyter notebook
project_root = Path().resolve().parent  # Assuming notebook is in tests directory
sys.path.append('')

# Set up logging
logging.basicConfig(level=logging.INFO)


# Import KnowledgeSynthesis
from src.core.synthesis import KnowledgeSynthesis

def create_sample_data():
    """Create sample data sources of different types"""
    
    # Create directories for sample data
    data_dir = Path("tests/data/synthesis")
    data_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Create raster data
    raster_path = data_dir / "sample_raster.npy"
    x, y = np.mgrid[0:100, 0:100]
    raster_data = np.stack([
        np.sin(x/10) + np.cos(y/10),  # Band 1
        np.sin(x/20) + np.cos(y/20),  # Band 2
        np.sin(x/30) + np.cos(y/30)   # Band 3
    ], axis=-1)
    np.save(raster_path, raster_data)
    
    # 2. Create vector data
    vector_path = data_dir / "sample_vector.parquet"
    import pandas as pd
    vector_data = pd.DataFrame({
        'feature1': np.random.normal(0, 1, 1000),
        'feature2': np.random.normal(2, 1.5, 1000),
        'feature3': np.random.exponential(2, 1000)
    })
    vector_data.to_parquet(vector_path)
    
    # 3. Create time series data
    timeseries_path = data_dir / "sample_timeseries.nc"
    import xarray as xr
    times = pd.date_range('2024-01-01', '2024-01-31', freq='D')
    timeseries_data = xr.Dataset(
        data_vars={
            'temperature': (('time'), np.sin(np.arange(len(times)) / 5) + np.random.normal(0, 0.1, len(times))),
            'humidity': (('time'), np.cos(np.arange(len(times)) / 5) + np.random.normal(0, 0.1, len(times)))
        },
        coords={
            'time': times
        }
    )
    timeseries_data.to_netcdf(timeseries_path)
    
    # Create data source descriptions
    data_sources = [
        {
            "type": "raster",
            "path": str(raster_path),
            "bands": ["band1", "band2", "band3"]
        },
        {
            "type": "vector",
            "path": str(vector_path)
        },
        {
            "type": "timeseries",
            "path": str(timeseries_path)
        }
    ]
    
    return data_sources

def visualize_patterns(patterns, title):
    """Visualize synthesis patterns"""
    plt.figure(figsize=(12, 4))
    
    # Plot feature importance
    plt.subplot(131)
    plt.bar(range(len(patterns["feature_importance"])), 
            patterns["feature_importance"])
    plt.title("Feature Importance")
    plt.xlabel("Feature Index")
    plt.ylabel("Importance")
    
    # Plot cluster assignments
    plt.subplot(132)
    unique_clusters = np.unique(patterns["clusters"])
    plt.hist(patterns["clusters"], bins=len(unique_clusters))
    plt.title("Cluster Distribution")
    plt.xlabel("Cluster")
    plt.ylabel("Count")
    
    # Plot cluster centers
    plt.subplot(133)
    centers = np.array(patterns["cluster_centers"])
    plt.imshow(centers, aspect='auto')
    plt.title("Cluster Centers")
    plt.xlabel("Feature")
    plt.ylabel("Cluster")
    plt.colorbar()
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

def main():
    # Create sample data
    print("Creating sample data sources...")
    data_sources = create_sample_data()
    
    # Initialize synthesis system
    synthesis = KnowledgeSynthesis()
    
    # Set synthesis parameters
    parameters = {
        "n_clusters": 3,
        "random_state": 42,
        "min_samples": 10
    }
    
    # Run synthesis
    print("\nRunning knowledge synthesis...")
    try:
        result = synthesis.synthesize(
            data_sources=data_sources,
            parameters=parameters
        )
        
        # Print results
        print("\nSynthesis Results:")
        print("-" * 30)
        print(f"Memory ID: {result['memory_id']}")
        print(f"Number of patterns found: {len(result['patterns'])}")
        print(f"Number of insights generated: {len(result['insights'])}")
        
        # Visualize patterns
        visualize_patterns(result["patterns"], "Synthesis Patterns")
        
        # Print insights
        print("\nGenerated Insights:")
        for insight in result["insights"]:
            print(f"\nType: {insight['type']}")
            print(f"Description: {insight['description']}")
            for key, value in insight.items():
                if key not in ['type', 'description']:
                    print(f"{key}: {value}")
        
    except Exception as e:
        print(f"Error during synthesis: {e}")
        raise



In [ ]:
main()